<img src="https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/IMPERIAL_logo_RGB_Blue_2024.svg" alt="Imperial Logo" width="500"/><br /><br />

ML Foundations and Data Preparation - Preparation Notebook
==============
### SURG70098 - Surgical Data Science and AI
### Stuart Bowyer

## How to use this notebook

**Do this on your own before Session 4. It should take 45 to 60 minutes.** Session 4 does not re-teach these methods: it starts from them, and asks when each one helps and when it misleads.

Each block follows the same pattern:

1. **Read** the short explanation and **run** the worked example.
2. **» Modify** — change a working cell so it answers the next question, then run the hidden **check** cell below it. It prints **Correct**, or **Not yet** with a pointer.
3. **Compare** — run a cell where the same method gives two different answers, and work out why before opening **What to notice**.

Run the cells in order: each block builds on the tables above it. After a restart, use **Runtime → Run before** to catch up.

Stuck? The pandas documentation or an AI assistant will answer questions about the code. **Never paste MIMIC rows or outputs into an AI assistant, or share them**: the data use agreement forbids it.

## Contents
1. Exploring one variable
2. Exploring two variables
3. Finding and dropping missing data
4. Filling missing data
5. Fixing types
6. Outliers and invalid values

The last cell has a one-page summary of the methods.

## Setup
This uses the same course views as Session 3. The first time, Colab asks you to sign in: use the **Google account linked to PhysioNet**. Loading takes a minute or two, so **run it once**.

In [ ]:
%pip install pandas_gbq --quiet
import pandas as pd
import matplotlib.pyplot as plt
import pandas_gbq

# The Google Cloud project that pays for your queries. After the course, use your own.
PROJECT = 'bowyerlab-mimic-teaching'

def query(sql, project=PROJECT):
    """Run a SQL query on BigQuery and return the result as a DataFrame."""
    return pandas_gbq.read_gbq(sql, project_id=project)

def check(passed, hint):
    print('Correct' if passed else 'Not yet: ' + hint)

# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.patients
df_patients = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.patients`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.admissions
df_admissions = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.admissions`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_icu.chartevents, keeping itemid 220045
df_hr = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.chartevents_hr`")

print(len(df_patients), len(df_admissions), len(df_hr))   # 6549 8647 898404

**`df_stays`** is the table you will use throughout: one row per ICU admission. It adds three columns to what you met in Session 3: `died` (in hospital), `hr_mean_24h` and `hr_max_24h` (the mean and maximum heart rate in the first 24 hours after admission).

In [ ]:
df_stays = df_admissions.merge(df_patients, on='subject_id')
stay_length = df_stays['dischtime'] - df_stays['admittime']
df_stays['los_days'] = stay_length.dt.total_seconds() / (60 * 60 * 24)
df_stays['died'] = df_stays['hospital_expire_flag'].astype(int)
df_stays['anchor_age'] = df_stays['anchor_age'].astype(int)
df_stays['anchor_year'] = df_stays['anchor_year'].astype(int)

# Heart rate readings in the first 24 hours of each admission
df_timed = df_hr.merge(df_admissions[['hadm_id', 'admittime']], on='hadm_id')
time_since_admission = df_timed['charttime'] - df_timed['admittime']
df_timed['hours'] = time_since_admission.dt.total_seconds() / (60 * 60)
df_first24h = df_timed[(df_timed['hours'] >= 0) & (df_timed['hours'] <= 24)]

hr_24h = df_first24h.groupby('hadm_id')['valuenum'].agg(hr_mean_24h='mean', hr_max_24h='max')
hr_24h = hr_24h.reset_index()
df_stays = df_stays.merge(hr_24h, on='hadm_id', how='left')

print(df_stays.shape)

## 1. Exploring one variable
**Exploratory data analysis (EDA)** means looking at your data before you model it. Its tools sort into a simple grid:

| | **Numbers** | **Pictures** |
| :-- | :-- | :-- |
| **One variable** | `describe`, `value_counts` | histogram, box plot |
| **Two or more variables** | `corr`, `crosstab` | scatter plot, box plot by group |

Numbers and pictures each hide something the other shows, so use both. This block covers one variable; the next covers two.

In [ ]:
df_stays['los_days'].describe()

In [ ]:
df_stays['gender'].value_counts(normalize=True)   # normalize=True gives proportions, not counts

In [ ]:
df_stays['los_days'].plot.hist(bins=50)
plt.title('Most hospital stays are short, with a long tail')
plt.xlabel('Length of stay (days)')
plt.ylabel('Admissions')
plt.show()

df_stays.boxplot(column='los_days')
plt.ylabel('Length of stay (days)')
plt.show()

**» Modify** — Change this so `shares` holds the proportions of **`admission_type`** instead. Which type is most common?

In [ ]:
shares = df_stays['gender'].value_counts(normalize=True)
shares

In [ ]:
#@title Run to check your answer
check(len(shares) == df_stays['admission_type'].nunique() and abs(shares.sum() - 1) < 1e-9,
      'keep normalize=True, and change the column')

**Compare** — Two summaries of a typical stay, and the shortest stay. Which of the first two would you report, and what is odd about the third?

In [ ]:
print('Mean:  ', df_stays['los_days'].mean())
print('Median:', df_stays['los_days'].median())
print('Min:   ', df_stays['los_days'].min())

<details><summary><b>What to notice</b></summary>

The mean (about 10 days) is well above the median (about 7), because the long tail pulls it up. The median is closer to a typical stay.

The shortest stay is **negative**. Six admissions were discharged before they were admitted, according to the data. Nobody finds that without looking.

</details>

## 2. Exploring two variables
`corr` measures how closely two numeric columns move together in a straight line: 1 is a perfect positive relationship, 0 none, and -1 a perfect negative one.

In [ ]:
df_stays[['anchor_age', 'los_days', 'hr_mean_24h', 'hr_max_24h']].corr()

Mean and maximum heart rate move together, as you would expect. Most of the others are close to 0, but **close to 0 does not mean unrelated**: `corr` only measures straight lines. Median stay rises from about 6 days under 40 to over 7 at 65 to 80, then falls again over 80, so age and length of stay are related, in a curve that `corr` cannot see.

`crosstab` counts each combination of two categories.

In [ ]:
pd.crosstab(df_stays['gender'], df_stays['died'], normalize='index')   # proportions within each row

In [ ]:
df_stays.plot.scatter(x='anchor_age', y='hr_mean_24h', s=2)
plt.title('Heart rate against age')
plt.xlabel('Age (years)')
plt.ylabel('Mean heart rate, first 24 h (bpm)')
plt.show()

df_stays.boxplot(column='los_days', by='gender')
plt.suptitle('')
plt.title('Length of stay by gender')
plt.ylabel('Length of stay (days)')
plt.ylim(0, 60)
plt.show()

A **scatter plot matrix** draws every pair of columns at once, which is a quick first look at a table with several numeric columns. Each column gets a row and a column in the grid. The diagonal, where a column meets itself, shows its histogram; every other panel is a scatter plot of that pair. The panels either side of the diagonal show the same data, with the axes swapped.

In [ ]:
pd.plotting.scatter_matrix(df_stays[['anchor_age', 'los_days']], figsize=(6, 6))
plt.show()

**» Modify** — Change this so `mortality_table` shows in-hospital death by **`admission_type`** instead of gender. Which types are lowest?

In [ ]:
mortality_table = pd.crosstab(df_stays['gender'], df_stays['died'], normalize='index')
mortality_table

In [ ]:
#@title Run to check your answer
check(len(mortality_table) == df_stays['admission_type'].nunique() and abs(mortality_table.sum(axis=1) - 1).max() < 1e-9,
      'change the first column, and keep normalize=\'index\'')

**Compare** — The correlation between the mean and maximum heart rate in the first 24 hours, on all the admissions, then leaving out any with a maximum above 250 bpm. How many admissions did that remove, and why did the answer move so much?

In [ ]:
plausible = df_stays[df_stays['hr_max_24h'] <= 250]

print('All admissions: ', df_stays['hr_mean_24h'].corr(df_stays['hr_max_24h']))
print('Maximum <= 250: ', plausible['hr_mean_24h'].corr(plausible['hr_max_24h']))
print('Admissions removed:', df_stays['hr_max_24h'].notna().sum() - len(plausible))

<details><summary><b>What to notice</b></summary>

Removing just **two** admissions out of nearly 7,000 moves the correlation from about 0.56 to about 0.85. Their maximum heart rates are 950 and 10,109 bpm: entry errors, not measurements.

A single number can be wrecked by a handful of rows. A scatter plot would have shown you them at once.

</details>

## 3. Finding and dropping missing data
Pandas marks missing values as `NaN` (or `NaT` for dates). `.isna()` finds them, and because `True` counts as 1, `.isna().sum()` counts them per column.

In [ ]:
df_stays.isna().sum()

`.dropna()` removes rows with missing values, and `subset=` says which columns to look at. `.drop(columns=...)` removes a whole column.

In [ ]:
print(len(df_stays))
print(len(df_stays.dropna(subset=['marital_status'])))    # drop the rows missing this column
print(df_stays.drop(columns=['marital_status']).shape)    # drop the whole column

**» Modify** — Change this so `df_kept` keeps only the admissions that have a value for **`hr_mean_24h`**. How many are left?

In [ ]:
df_kept = df_stays.dropna(subset=['marital_status'])
len(df_kept)

In [ ]:
#@title Run to check your answer
check(len(df_kept) == df_stays['hr_mean_24h'].notna().sum(), 'change the column inside subset=[...]')

**Compare** — The obvious shortcut: `.dropna()` with no `subset`, which drops any row with a missing value in any column. How many admissions survive, and what proportion of those patients died in hospital?

In [ ]:
df_complete = df_admissions.dropna()

print('Admissions:', len(df_admissions), 'then', len(df_complete))
print('Died:      ', df_admissions['hospital_expire_flag'].mean(), 'then', df_complete['hospital_expire_flag'].mean())

<details><summary><b>What to notice</b></summary>

Every admission left is one where the patient **died**. `deathtime` is missing for everyone who survived, so "complete rows only" silently removed all the survivors.

Here a missing value is information: it means the patient was alive. Before you drop or fill anything, ask why it is missing. We pick that up in Session 4.

</details>

## 4. Filling missing data
**Imputation** replaces a missing value with a substitute. `.fillna()` does it, and the simplest substitute is a constant.

In [ ]:
marital_filled = df_stays['marital_status'].fillna('UNKNOWN')
marital_filled.value_counts()

It can also fill with a summary of the values that are present, such as the mean or the median. There are more advanced methods too, such as predicting the value from other columns; Session 4 discusses how to choose.

In [ ]:
hr = df_stays['hr_mean_24h']
filled = hr.fillna(hr.mean())
filled.isna().sum()      # nothing left missing

**» Modify** — Change this so the missing values in `filled` are replaced with the **median** instead of the mean.

In [ ]:
filled = hr.fillna(hr.mean())
filled.isna().sum()

In [ ]:
#@title Run to check your answer
check(filled.isna().sum() == 0 and abs(filled[hr.isna()] - hr.median()).max() < 1e-9, 'use .median() where the cell has .mean()')

**Compare** — Filling assumes the missing admissions are like the others. Are they? `hr_is_missing` is `True` for the admissions with no heart rate in the first 24 hours. Compare their length of stay, then their in-hospital deaths.

In [ ]:
df_stays['hr_is_missing'] = hr.isna()
df_stays.groupby('hr_is_missing')['los_days'].median()

In [ ]:
df_stays.groupby('hr_is_missing')['died'].mean()

<details><summary><b>What to notice</b></summary>

They are different: a median stay of 11 days rather than about 6, and 15% dying in hospital rather than about 11%.

Every one of those admissions has its first charted heart rate later, typically about three days after admission. Heart rate here comes from ICU charting, so these are mostly patients who reached the ICU later. Giving them the average heart rate treats them as average patients, which they are not.

</details>

## 5. Fixing types
Data often arrive as text. This block uses a small made-up table of messy records, because the MIMIC views are already typed.

In [ ]:
DATA = "https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/data/"
df_messy = pd.read_csv(DATA + "messy_data.csv")
df_messy[['Age', 'Weight', 'BloodPressure', 'Temp']]

The `.str` methods from Session 2 clean a text column, and `.astype()` converts it once it is clean.

In [ ]:
df_bp = df_messy[['BloodPressure']].dropna()
parts = df_bp['BloodPressure'].str.split('/', expand=True)
df_bp['systolic'] = parts[0].astype(int)
df_bp

**» Modify** — Change this so it makes a **`diastolic`** column instead, from the number after the `/`.

In [ ]:
parts = df_bp['BloodPressure'].str.split('/', expand=True)
df_bp['systolic'] = parts[0].astype(int)
df_bp

In [ ]:
#@title Run to check your answer
check('diastolic' in df_bp and (df_bp['diastolic'] == df_bp['BloodPressure'].str.split('/').str[1].astype(int)).all(),
      '[0] is the first part of the split: which is the second? Then rename the new column')

**Compare** — Two ways of getting numbers out of text columns. Each runs without an error. Are the results right?

First, `pd.to_numeric(..., errors='coerce')`, which turns anything it cannot convert into `NaN`.

In [ ]:
weight = pd.to_numeric(df_messy['Weight'], errors='coerce')
print('Weights kept:', weight.notna().sum(), 'of', len(weight))
print(pd.DataFrame({'text': df_messy['Weight'], 'number': weight}))

Second, stripping the letters with `.str.replace`, then converting with `.astype`.

In [ ]:
temp_text = df_messy['Temp'].str.replace('[FC ]', '', regex=True)
temp = temp_text.astype(float)
print('Mean temperature:', temp.mean())
print(pd.DataFrame({'text': df_messy['Temp'], 'number': temp}))

<details><summary><b>What to notice</b></summary>

`errors='coerce'` kept only the weights with no unit, and silently threw the rest away. The two it kept (130 and 165.3) have no unit at all, so you cannot tell kilograms from pounds.

Stripping the letters from `Temp` gives clean-looking numbers, but they mix Fahrenheit and Celsius, so the mean describes nobody. **A successful conversion is not a correct one**: check the values, not just the type.

</details>

## 6. Outliers and invalid values
An **invalid** value is impossible; an **outlier** is unusual, and may be real. There are three common ways to find them, all tried here on every heart rate reading.

In [ ]:
readings = df_hr['valuenum']
readings.describe()

**Clinical limits:** you decide what is possible.

In [ ]:
invalid = (readings < 20) | (readings > 250)
print('Outside limits:', invalid.sum())

**Z-score:** how many standard deviations a value is from the mean, the same idea as the Z-score on a bone density report. In a normal distribution, about 99.7% of values fall within 3 standard deviations, so more than 3 is a common cut-off for an outlier.

In [ ]:
z = (readings - readings.mean()) / readings.std()
flagged = z.abs() > 3
print('Z-score above 3:', flagged.sum())

**Interquartile range (IQR):** the range of the middle 50% of values. Often, more than 1.5 IQRs beyond it counts as an outlier.

In [ ]:
q1, q3 = readings.quantile(0.25), readings.quantile(0.75)
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print('IQR limits:', low, high)
print('Beyond them:', ((readings < low) | (readings > high)).sum())

`.mask(condition)` replaces the values where the condition is true with `NaN`, which keeps the row.

In [ ]:
clean = readings.mask(invalid)
print('Largest after masking:', clean.max())

**» Modify** — Change this so `z_clean` is the z-score of the **`clean`** readings, with the invalid values already masked, instead of the raw `readings`. How many readings are above 3 now?

In [ ]:
z_clean = (readings - readings.mean()) / readings.std()
flagged_clean = z_clean.abs() > 3
flagged_clean.sum()

In [ ]:
#@title Run to check your answer
expected = (clean - clean.mean()) / clean.std()
check(abs(z_clean.std() - expected.std()) < 1e-9 and (z_clean.abs() > 3).sum() == (expected.abs() > 3).sum(),
      'use clean in all three places, not readings')

**Compare** — Three methods on the same readings: look back at their counts above. Why does the z-score find so few on the raw readings, and many more once they are clean? And what are the readings that the IQR flags?

In [ ]:
print('Standard deviation, raw:  ', readings.std())
print('Standard deviation, clean:', clean.std())
print('Largest reading:          ', readings.max())

flagged_by_iqr = readings[(readings > high) & (readings <= 250)]
print('Flagged by IQR and possible:', len(flagged_by_iqr))
print(flagged_by_iqr.describe())

<details><summary><b>What to notice</b></summary>

**The z-score is blinded by the errors it should catch.** One reading of 5 million inflates the standard deviation to over 5,000 bpm, so almost nothing else is more than 3 standard deviations away: it flags 2 readings of nearly 900,000, while 429 are outside the clinical limits. Masked first, the z-score flags about 5,100.

**The IQR is not fooled, but it does not know medicine.** It flags about 10,400 readings, and most are heart rates above about 135 bpm: fast, but real, and exactly what you would not want to delete from ICU data.

Statistics find unusual values. Only context says which are wrong.

</details>

## Summary of the methods

| Task | Pandas | Watch out for |
| :-- | :-- | :-- |
| Summarise one variable | `describe`, `value_counts(normalize=True)`, `.plot.hist`, `.boxplot` | A mean pulled by a long tail |
| Compare two variables | `corr`, `pd.crosstab`, `.plot.scatter`, `.boxplot(by=...)` | One or two rows wrecking a correlation |
| Find and drop missing data | `isna().sum()`, `dropna(subset=...)`, `drop(columns=...)` | Rows dropped for a reason that matters |
| Fill missing data | `fillna` with a constant, the mean or the median | Missing values unlike the others |
| Fix types | `astype`, `.str` methods, `pd.to_numeric(errors='coerce')` | A conversion that runs but is wrong |
| Find invalid values and outliers | Limits, z-score, IQR, `.mask` | Errors hiding from the z-score; real values flagged by the IQR |

**Bring to Session 4:** the *Compare* that surprised you most.